# Build a Forecasting Model

In the EDA notebook we got a feel for the Walmart sales data. Now we put
that understanding to work and build a **weekly sales forecast** at the
`(Store, Department)` level.

We will go through the same workflow you would follow on a real
forecasting project:

1. **Load and merge** the source files.
2. **Define the forecast horizon**: how far ahead we forecast, and what we
   know at the moment we make the forecast.
3. **Engineer features** such as calendar parts, lag and rolling features,
   and store attributes, using only information available at forecast time.
4. **Split** the data using a **time-based** holdout (no random shuffles).
5. **Build two naive baselines** so we know what "good" looks like.
6. **Train a Ridge regression** as a simple, interpretable benchmark.
7. **Train a gradient-boosted tree (XGBoost)** model.
8. **Evaluate** with MAE, RMSE, and the competition's WMAE, overall and by
   weeks ahead.
9. **Interpret** the results and discuss how to improve them.

## Setup


In [1]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio

from sklearn.compose import ColumnTransformer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error, mean_squared_error
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import xgboost as xgb

pio.templates.default = "simple_white"

print("pandas  :", pd.__version__)
print("numpy   :", np.__version__)
print("xgboost :", xgb.__version__)

pandas  : 3.1.1
numpy   : 2.5.4
xgboost : 3.4.1


## Load and Merge the Data

We pull the same three files we used in EDA and join them into a single
modeling table.


In [2]:
BASE = "https://raw.githubusercontent.com/bdi593/datasets/main/walmart-store-sales-forecasting"

df_train = pd.read_csv(f"{BASE}/train.csv", parse_dates=["Date"])
df_features = pd.read_csv(f"{BASE}/features.csv", parse_dates=["Date"])
df_stores = pd.read_csv(f"{BASE}/stores.csv")

df = (
    df_train.merge(df_features, on=["Store", "Date", "IsHoliday"], how="left")
    .merge(df_stores, on="Store", how="left")
    .sort_values(["Store", "Dept", "Date"])
    .reset_index(drop=True)
)

print("rows:", len(df), "  cols:", df.shape[1])
df.head()

rows: 421570   cols: 16


,Store,Dept,Date,Weekly_Sales,IsHoliday,Temperature,Fuel_Price,MarkDown1,MarkDown2,MarkDown3,MarkDown4,MarkDown5,CPI,Unemployment,Type,Size
0,1,1,2010-02-05,24924.50,False,42.31,2.572,NaN,NaN,NaN,NaN,NaN,211.096358,8.106,A,151315
1,1,1,2010-02-12,46039.49,True,38.51,2.548,NaN,NaN,NaN,NaN,NaN,211.242170,8.106,A,151315
2,1,1,2010-02-19,41595.55,False,39.93,2.514,NaN,NaN,NaN,NaN,NaN,211.289143,8.106,A,151315
3,1,1,2010-02-26,19403.54,False,46.63,2.561,NaN,NaN,NaN,NaN,NaN,211.319643,8.106,A,151315
4,1,1,2010-03-05,21827.90,False,46.50,2.625,NaN,NaN,NaN,NaN,NaN,211.350143,8.106,A,151315


## Define the Forecast Horizon

Before engineering any features we need to decide **what question the
forecast answers**. Here the question is: _standing at the end of the
training data, what will each department sell in each of the next 12
weeks?_

- The last week of training data is the **forecast origin**.
- The 12 weeks after it are the **forecast horizon**. They are also our
  test set.

This choice shapes every feature below. A feature can only be used if its
value is known at the forecast origin. Last week's actual sales are known
when we forecast next week, but not when we forecast 12 weeks out.

In [3]:
HORIZON = 12  # number of weeks we forecast at once

cutoff = df["Date"].max() - pd.Timedelta(weeks=HORIZON)
print("forecast origin (last training week):", cutoff.date())
print(
    "forecast horizon:",
    (cutoff + pd.Timedelta(weeks=1)).date(),
    "to",
    df["Date"].max().date(),
)

forecast origin (last training week): 2012-08-03
forecast horizon: 2012-08-10 to 2012-10-26


:::{note} Why not use last week's sales as a feature?

A common shortcut is a `Sales_Lag1` feature: last week's actual sales.
Inside a 12-week test window, that feature quietly uses the actual sales of
weeks 1 to 11 of the window, which nobody knows at the forecast origin. The
model is then scored as a **one-week-ahead** forecast while we describe it
as a 12-week forecast.

One-week-ahead forecasts are legitimate if the business only needs next
week's number. In that case, say so, and compare the model with a
**persistence** forecast that simply repeats last week's sales. Here we
want the full 12 weeks, so every history feature must look back at least
`HORIZON` weeks.

:::

## Feature Engineering

The two biggest gains in retail forecasting almost always come from:

1. **Calendar features** that let the model learn seasonality.
2. **Lag and rolling features** that let the model use the history of
   the _same_ `(Store, Dept)` series as a predictor.

We also keep store **type** and **size** as static features.

### Calendar features


In [4]:
df["Year"] = df["Date"].dt.year
df["Month"] = df["Date"].dt.month
df["WeekOfYear"] = df["Date"].dt.isocalendar().week.astype(int)
df["IsHoliday"] = df["IsHoliday"].astype(int)

# Rough flags for the four key holidays (dates given by the dataset).
super_bowl = ["2010-02-12", "2011-02-11", "2012-02-10"]
labor_day = ["2010-09-10", "2011-09-09", "2012-09-07"]
thanksgiving = ["2010-11-26", "2011-11-25"]
christmas = ["2010-12-31", "2011-12-30"]

df["IsSuperBowl"] = df["Date"].isin(pd.to_datetime(super_bowl)).astype(int)
df["IsLaborDay"] = df["Date"].isin(pd.to_datetime(labor_day)).astype(int)
df["IsThanksgiving"] = df["Date"].isin(pd.to_datetime(thanksgiving)).astype(int)
df["IsChristmas"] = df["Date"].isin(pd.to_datetime(christmas)).astype(int)

### Lag and rolling features

For each `(Store, Dept)` series we add three history features. Each one
looks back at least `HORIZON` (12) weeks, so it is known at the forecast
origin for every week of the horizon:

- `Sales_Lag12`: sales 12 weeks earlier. For the last week of the horizon
  this is the forecast origin itself, the most recent week we know.
- `Sales_Lag52`: sales 52 weeks earlier (the same week last year), a
  natural seasonal benchmark.
- `Sales_Roll4_Lag12`: the mean of the weeks 12 to 15 weeks earlier, a
  smoother version of `Sales_Lag12`. If some of those four weeks are
  missing, it averages the ones that exist.

One detail matters here. `groupby(...).shift(52)` moves values by 52
**rows**, not 52 **weeks**. If a series is missing some weeks, a row shift
reaches back further than a year. So we first reindex every series to a
complete weekly calendar, shift on that calendar, and then join the lag
features back to the rows we observe.

In [5]:
weeks_per_series = df.groupby(["Store", "Dept"]).size()
n_weeks = df["Date"].nunique()
print(
    f"series with fewer than {n_weeks} weekly rows: "
    f"{(weeks_per_series < n_weeks).sum():,} of {len(weeks_per_series):,}"
)

# 1. A complete weekly calendar for every (Store, Dept) series.
all_weeks = pd.date_range(df["Date"].min(), df["Date"].max(), freq="7D")
calendar = (
    df[["Store", "Dept"]]
    .drop_duplicates()
    .merge(pd.DataFrame({"Date": all_weeks}), how="cross")
    .merge(
        df[["Store", "Dept", "Date", "Weekly_Sales"]],
        on=["Store", "Dept", "Date"],
        how="left",
    )
    .sort_values(["Store", "Dept", "Date"])
)

# 2. Shift on the calendar, so shift(k) always means "k weeks earlier".
sales = calendar.groupby(["Store", "Dept"])["Weekly_Sales"]
calendar["Sales_Lag12"] = sales.shift(HORIZON)
calendar["Sales_Lag52"] = sales.shift(52)
calendar["Sales_Roll4_Lag12"] = sales.transform(
    lambda s: s.shift(HORIZON).rolling(4, min_periods=1).mean()
)

# 3. Join the lag features back to the observed rows.
lag_cols = ["Sales_Lag12", "Sales_Lag52", "Sales_Roll4_Lag12"]
df = df.merge(
    calendar[["Store", "Dept", "Date", *lag_cols]],
    on=["Store", "Dept", "Date"],
    how="left",
)

# How often would a plain row shift have picked the wrong week?
row_shift_lag52 = df.groupby(["Store", "Dept"])["Weekly_Sales"].shift(52)
has_row_lag = row_shift_lag52.notna()
wrong_week = (row_shift_lag52[has_row_lag] != df.loc[has_row_lag, "Sales_Lag52"]).mean()
print(f"row-shift lag-52 values that are not from 52 weeks earlier: {wrong_week:.1%}")

series with fewer than 143 weekly rows: 671 of 3,331


row-shift lag-52 values that are not from 52 weeks earlier: 5.0%


### Fill missing lag values using the training weeks only

The first year of each series has no `Sales_Lag52`, and the first 12 weeks
have no `Sales_Lag12`. Rather than drop those rows, we fill the gaps with
the series' **median weekly sales over the training weeks**. Series with
no training history at all get the overall training median.

The fill values must come from the training period only. A median over all
weeks would include the test weeks, and their sales would leak into the
features. (For early training rows, the training median still includes
some later training weeks. That is a mild look-ahead inside the training
data, but it reveals nothing about the test window.)

In [6]:
is_train = df["Date"] <= cutoff
print("share of training rows with a missing lag feature:")
print(df.loc[is_train, lag_cols].isna().mean().round(3))

# Sales from the training weeks only; test weeks become NaN and are ignored.
train_sales = df["Weekly_Sales"].where(is_train)
overall_median = train_sales.median()
fill_values = (
    train_sales.groupby([df["Store"], df["Dept"]])
    .transform("median")
    .fillna(overall_median)  # series with no training weeks
)

for col in lag_cols:
    df[col] = df[col].fillna(fill_values)

share of training rows with a missing lag feature:
Sales_Lag12          0.111
Sales_Lag52          0.412
Sales_Roll4_Lag12    0.101
dtype: float64


### Markdowns, CPI, and unemployment

The EDA showed that the `MarkDown` columns were not recorded at all before
November 2011. After that date some values are still missing; the dataset
description says markdown data "is not available for all stores all the
time". We fill every gap with 0 and add a `MarkDown_Recorded` flag, so the
model can tell the weeks before markdowns were recorded from the weeks
after. Read a 0 as "no markdown recorded", not as proof that no markdown
ran.

`CPI` and `Unemployment` are missing in `features.csv` only for weeks
after the last week of sales, so they need no filling here.

In [7]:
markdown_cols = [f"MarkDown{i}" for i in range(1, 6)]
markdown_start = df_features.loc[
    df_features[markdown_cols].notna().any(axis=1), "Date"
].min()
print("markdowns first recorded:", markdown_start.date())

df["MarkDown_Recorded"] = (df["Date"] >= markdown_start).astype(int)
df[markdown_cols] = df[markdown_cols].fillna(0)

# Any missing values left?
df.isna().sum().sort_values(ascending=False).head()

markdowns first recorded: 2011-11-11


Store           0
Dept            0
Date            0
Weekly_Sales    0
IsHoliday       0
dtype: int64

## Train / Test Split

This is the most important step in any forecasting project. **You must
split by time**, not at random. If we shuffle, future weeks leak into the
training set and we end up with optimistic, useless metrics.

We hold out the 12 weeks of the forecast horizon as our test set and
train on everything up to the forecast origin.

In [8]:
train = df[df["Date"] <= cutoff].copy()
test = df[df["Date"] > cutoff].copy()

print(
    f"train rows: {len(train):>8,}  ({train['Date'].min().date()} → {train['Date'].max().date()})"
)
print(
    f"test  rows: {len(test):>8,}  ({test['Date'].min().date()} → {test['Date'].max().date()})"
)
print(
    "holiday weeks in the test window:",
    sorted(test.loc[test["IsHoliday"] == 1, "Date"].dt.strftime("%Y-%m-%d").unique()),
)

train rows:  386,007  (2010-02-05 → 2012-08-03)
test  rows:   35,563  (2012-08-10 → 2012-10-26)
holiday weeks in the test window: ['2012-09-07']


:::{caution} What this test window can and cannot tell us

The only holiday week in the test window is **Labor Day** (7 September
2012). Thanksgiving and Christmas, the weeks with the largest sales swings,
fall outside it. So the scores below say little about how well any model
handles the holiday season. To test that, you would hold out a window that
covers November and December 2011 and train on the weeks before it.

:::

## Two Naive Baselines

Before training anything fancy we build two forecasts that need no model
at all:

1. **Seasonal naive.** Predict each week's sales as the same week last
   year (`Sales_Lag52`). This is hard to beat in retail because so much
   demand is annual. Where a series has no sales 52 weeks earlier,
   `Sales_Lag52` already holds the series' training median from the fill
   step.
2. **Persistence.** Predict every week of the horizon as the last observed
   week of the series up to the forecast origin. This is the lag-1 naive
   forecast, made once at the origin and held flat for 12 weeks. Series
   with no training history get the overall training median.

In [9]:
def regression_metrics(y_true, y_pred, weights=None):
    """Return MAE, RMSE, and (optionally) WMAE in a tidy dict."""
    out = {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": float(np.sqrt(mean_squared_error(y_true, y_pred))),
    }
    if weights is not None:
        weights = np.asarray(weights, dtype=float)
        out["WMAE"] = float(
            np.sum(weights * np.abs(np.asarray(y_true) - np.asarray(y_pred)))
            / weights.sum()
        )
    return out


# Holiday weeks are weighted 5x in the Walmart competition metric (WMAE).
test_weights = np.where(test["IsHoliday"] == 1, 5.0, 1.0)

# Seasonal naive: the same week last year.
test["Pred_Seasonal"] = test["Sales_Lag52"]

# Persistence: the last observed week of each series, held flat.
last_observed = (
    train.sort_values("Date").groupby(["Store", "Dept"])["Weekly_Sales"].last()
)
test["Pred_Persistence"] = (
    test.join(last_observed.rename("Last_Observed"), on=["Store", "Dept"])[
        "Last_Observed"
    ].fillna(overall_median)
)

seasonal_metrics = regression_metrics(
    test["Weekly_Sales"], test["Pred_Seasonal"], weights=test_weights
)
persistence_metrics = regression_metrics(
    test["Weekly_Sales"], test["Pred_Persistence"], weights=test_weights
)
pd.DataFrame(
    {
        "Seasonal naive (lag-52)": seasonal_metrics,
        "Persistence (last observed week)": persistence_metrics,
    }
).T.round(2)

,MAE,RMSE,WMAE
Seasonal naive (lag-52),1687.13,3632.38,1713.35
Persistence (last observed week),2304.28,5069.11,2237.00


### How to read these numbers

- **MAE (Mean Absolute Error).** On average, the forecast is off by this
  many dollars per `(Store, Dept, Week)`. It is in the same units as the
  target, which makes it easy to talk to business stakeholders.
- **RMSE (Root Mean Squared Error).** Like MAE but penalizes large
  misses more heavily. If RMSE is much larger than MAE, a few big errors
  are dragging the score down.
- **WMAE (Weighted MAE).** Walmart's competition metric, identical to
  MAE except holiday weeks count 5×. In this test window the only holiday
  week is Labor Day, so WMAE stays close to MAE.

The seasonal naive forecast beats persistence on all three metrics. Over a
12-week horizon, last year's same week is a better guide than a single
value that grows staler every week. Any model we build needs to **beat the
seasonal naive numbers** to be worth using.

## Feature Set

We separate the columns into:

- **`numeric_features`**: quantities we can pass to a model directly.
- **`categorical_features`**: the store `Type` and the `Store` and `Dept`
  identifiers. Store 40 is not "more" than store 4, so these are labels,
  not numbers. We one-hot encode them for the linear model and pass them
  as categories to XGBoost.

:::{caution} Features we treat as known in advance

Calendar features, store attributes, and the lag features are genuinely
known at the forecast origin. `Temperature`, `Fuel_Price`, `CPI`,
`Unemployment`, and the `MarkDown` columns are not: we use their actual
values for the test weeks. That matches the Kaggle competition, which
supplied `features.csv` for the weeks to be forecast. In a real 12-week
forecast you would plug in forecasts or planned values for these columns,
or leave them out.

:::

In [10]:
numeric_features = [
    "Size",
    "Temperature",
    "Fuel_Price",
    "CPI",
    "Unemployment",
    *markdown_cols,
    "MarkDown_Recorded",
    "Year",
    "Month",
    "WeekOfYear",
    "IsHoliday",
    "IsSuperBowl",
    "IsLaborDay",
    "IsThanksgiving",
    "IsChristmas",
    *lag_cols,
]
categorical_features = ["Type", "Store", "Dept"]
target = "Weekly_Sales"

X_train = train[numeric_features + categorical_features]
y_train = train[target]
X_test = test[numeric_features + categorical_features]
y_test = test[target]

## Linear Regression Benchmark

A regularized linear model (Ridge) is a useful second benchmark: it is
fast, interpretable, and tells us how much of the signal simple linear
relationships can capture.

Two preprocessing steps matter for a linear model:

- **One-hot encode** the categorical features, so each store and
  department gets its own coefficient instead of a meaningless slope on
  its id number.
- **Standardize** the numeric features, so the Ridge penalty treats them
  on the same scale (`Size` is in the hundreds of thousands, `IsHoliday`
  is 0 or 1).

Putting both steps inside a `Pipeline` means they are fitted on the
training rows only.

In [11]:
preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), numeric_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ]
)

ridge = Pipeline(
    [
        ("prep", preprocess),
        ("model", Ridge(alpha=1.0)),
    ]
)

ridge.fit(X_train, y_train)
ridge_pred = ridge.predict(X_test)

ridge_metrics = regression_metrics(y_test, ridge_pred, weights=test_weights)
ridge_metrics

{'MAE': 1763.782448480922,
 'RMSE': 3415.655826106147,
 'WMAE': 1773.0673398630358}

## Gradient-Boosted Trees (XGBoost)

XGBoost is the workhorse of tabular forecasting. It handles non-linear
interactions, missing values, and large feature sets without much
hand-holding.


In [12]:
X_train_xgb = X_train.copy()
X_test_xgb = X_test.copy()
for col in categorical_features:
    # Use the training categories for both sets so the codes line up.
    categories = pd.CategoricalDtype(sorted(X_train[col].unique()))
    X_train_xgb[col] = X_train_xgb[col].astype(categories)
    X_test_xgb[col] = X_test_xgb[col].astype(categories)

model = xgb.XGBRegressor(
    n_estimators=600,
    learning_rate=0.05,
    max_depth=8,
    subsample=0.9,
    colsample_bytree=0.9,
    enable_categorical=True,
    tree_method="hist",
    random_state=42,
)

model.fit(X_train_xgb, y_train)
xgb_pred = model.predict(X_test_xgb)

xgb_metrics = regression_metrics(y_test, xgb_pred, weights=test_weights)
xgb_metrics

{'MAE': 1412.9410219941183,
 'RMSE': 2900.7694326385285,
 'WMAE': 1409.196307818153}

## Compare the Models


In [13]:
results = pd.DataFrame(
    {
        "Seasonal naive (lag-52)": seasonal_metrics,
        "Persistence (last observed week)": persistence_metrics,
        "Ridge regression": ridge_metrics,
        "XGBoost": xgb_metrics,
    }
).T.round(2)
results

,MAE,RMSE,WMAE
Seasonal naive (lag-52),1687.13,3632.38,1713.35
Persistence (last observed week),2304.28,5069.11,2237.00
Ridge regression,1763.78,3415.66,1773.07
XGBoost,1412.94,2900.77,1409.20


### Interpreting the comparison

- **XGBoost beats both baselines on every metric.** Its MAE is about 16%
  lower than the seasonal naive forecast's (roughly \$1,410 against
  \$1,690 per store-department-week).
- **Ridge does not beat the seasonal naive baseline on MAE or WMAE.** Its
  RMSE is lower, so it makes fewer very large misses, but its typical
  error is larger. One likely reason: a single linear equation shared by
  more than 3,000 series, fitted on raw dollar sales, cannot capture that
  large and small departments respond to the same features in different
  ways. Here the extra complexity of Ridge does not pay for itself.
- **WMAE is close to MAE for every method.** Only the Labor Day week
  carries extra weight in this window, so the competition metric adds
  little information here.
- **RMSE is about twice MAE for every method.** A minority of large misses
  accounts for much of the squared error. Investigating the worst
  predictions usually surfaces a missing feature, such as an unmodeled
  promotion.

### Error by weeks ahead

The table averages over all 12 weeks of the horizon. Splitting the error
by how many weeks ahead of the forecast origin each test week lies shows
how each method ages.

In [14]:
test["Pred_Ridge"] = ridge_pred
test["Pred_XGBoost"] = xgb_pred
test["WeeksAhead"] = (test["Date"] - cutoff).dt.days // 7

prediction_columns = {
    "Seasonal naive (lag-52)": "Pred_Seasonal",
    "Persistence (last observed week)": "Pred_Persistence",
    "Ridge regression": "Pred_Ridge",
    "XGBoost": "Pred_XGBoost",
}
mae_by_week = pd.DataFrame(
    {
        name: (test["Weekly_Sales"] - test[col]).abs().groupby(test["WeeksAhead"]).mean()
        for name, col in prediction_columns.items()
    }
)

fig = px.line(
    mae_by_week.reset_index().melt(
        id_vars="WeeksAhead", var_name="Model", value_name="MAE"
    ),
    x="WeeksAhead",
    y="MAE",
    color="Model",
    markers=True,
    title="Test MAE by weeks ahead of the forecast origin",
    labels={"WeeksAhead": "Weeks ahead", "MAE": "MAE ($ per store-department-week)"},
)
fig.update_xaxes(dtick=1)
fig.show()

mae_by_week.round(0)

,Seasonal naive (lag-52),Persistence (last observed week),Ridge regression,XGBoost
WeeksAhead,,,,
1,1624.0,1270.0,1653.0,1138.0
2,1665.0,1639.0,1703.0,1287.0
3,1944.0,2192.0,1940.0,1622.0
4,1972.0,2048.0,2040.0,1661.0
5,1792.0,2035.0,1801.0,1398.0
6,1375.0,2240.0,1540.0,1229.0
7,1625.0,2478.0,1611.0,1347.0
8,1587.0,2758.0,1612.0,1387.0
9,1719.0,2455.0,1810.0,1490.0


**Takeaway.**

- **Persistence ages fast.** It beats the seasonal naive forecast in the
  first two weeks of the horizon, when the last observed week is still
  recent. After that its error climbs, and by week 12 it is almost twice
  the seasonal naive error.
- **Seasonal naive does not age.** Last year's value is equally available
  for every week, so its error has no upward trend across the horizon.
- **XGBoost has the lowest error in every week.** Its lead over the
  seasonal naive forecast is largest in week 1 and smaller in most later
  weeks.
- Weeks 3 and 4 (the last two weeks of August) are harder than their
  neighbours for the seasonal naive forecast, Ridge, and XGBoost. A
  week-by-week error chart like this is a good place to start looking for
  a missing feature (for example, back-to-school timing).

## Feature Importance


In [15]:
importances = (
    pd.DataFrame(
        {"feature": X_train_xgb.columns, "importance": model.feature_importances_}
    )
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

fig = px.bar(
    importances,
    x="importance",
    y="feature",
    orientation="h",
    title="XGBoost feature importance",
    template="simple_white",
    height=600,
)
fig.update_yaxes(autorange="reversed")
fig.show()

importances.head(10)

,feature,importance
0,Sales_Lag52,0.682845
1,Sales_Roll4_Lag12,0.150571
2,IsThanksgiving,0.036099
3,Sales_Lag12,0.016123
4,MarkDown_Recorded,0.012878
5,WeekOfYear,0.011977
6,MarkDown3,0.011713
7,Dept,0.011373
8,IsChristmas,0.009319
9,Month,0.008662


**Takeaway.** The importance shown here is XGBoost's default for tree
models, **gain**: how much a feature improves the fit, on average, when a
tree splits on it.

- `Sales_Lag52` dominates, followed by `Sales_Roll4_Lag12`. Last year's
  same week and the recent level of the series carry most of the signal,
  which matches the strong yearly seasonality we saw in the EDA.
- `IsThanksgiving` comes next, even though the test window has no
  Thanksgiving. The model needs it to explain the training weeks.
- The `Store` and `Dept` identifiers rank low (`Dept` is just inside the
  top 10 and `Store` is not), because the lag features already tell the
  model each series' sales level.

Gain describes what the model relies on, not what causes sales.

## Visualizing the Forecast

Looking at predictions on a single series helps build intuition for where
the model wins and where it loses.


In [16]:
series = test.query("Store == 1 and Dept == 1").sort_values("Date")

plot_df = series.rename(
    columns={
        "Weekly_Sales": "Actual",
        "Pred_XGBoost": "XGBoost",
        "Pred_Seasonal": "Seasonal naive",
    }
).melt(
    id_vars="Date",
    value_vars=["Actual", "XGBoost", "Seasonal naive"],
    var_name="Series",
    value_name="Sales",
)

fig = px.line(
    plot_df,
    x="Date",
    y="Sales",
    color="Series",
    markers=True,
    title="Store 1, Dept 1: actual vs. forecasts (test window)",
)
fig.show()

For this series both forecasts follow the autumn rise in sales, because
last year's sales rose at the same time of year. Both overshoot the last
week of October by several thousand dollars: this year's increase was
smaller than last year's. XGBoost stays close to the seasonal naive line
for most of the window, adjusting it rather than replacing it.

### Residual Analysis

Where is the model wrong, and is it wrong in a structured way? The test
set has about 35,000 rows, so instead of handing every residual to a box
plot we compute the box-plot statistics first and draw the boxes from
them. Points beyond the whiskers are not drawn.

In [17]:
resid = test[["IsHoliday"]].assign(Residual=y_test - xgb_pred)


def box_stats(values):
    """Quartiles and Tukey whiskers (furthest points within 1.5 IQR)."""
    q1, median, q3 = values.quantile([0.25, 0.5, 0.75])
    iqr = q3 - q1
    return {
        "q1": q1,
        "median": median,
        "q3": q3,
        "lowerfence": values[values >= q1 - 1.5 * iqr].min(),
        "upperfence": values[values <= q3 + 1.5 * iqr].max(),
    }


resid_box = pd.DataFrame(
    {flag: box_stats(group) for flag, group in resid.groupby("IsHoliday")["Residual"]}
).T

fig = go.Figure(
    go.Box(
        x=resid_box.index.map({0: "Non-holiday weeks", 1: "Labor Day week"}),
        q1=resid_box["q1"],
        median=resid_box["median"],
        q3=resid_box["q3"],
        lowerfence=resid_box["lowerfence"],
        upperfence=resid_box["upperfence"],
        name="Residual",
    )
)
fig.update_layout(
    title="XGBoost residuals (actual - forecast): holiday vs. non-holiday weeks",
    yaxis_title="Residual ($)",
)
fig.show()

resid.groupby("IsHoliday")["Residual"].agg(
    rows="count",
    mean="mean",
    median="median",
    MAE=lambda r: r.abs().mean(),
).round(0)

,rows,mean,median,MAE
IsHoliday,,,,
0,32597,-160.0,-60.0,1414.0
1,2966,-61.0,-99.0,1398.0


**Takeaway.** In both groups the residuals are centred close to zero (the
median residual is slightly negative, so the model forecasts a little too
high for a typical row), and the Labor Day week has about the same spread
and MAE as the other weeks.

That is reassuring but weak evidence. It is a single holiday week, and
Labor Day weeks sell about as much as an ordinary week (see the EDA).
Whether the model handles Thanksgiving and Christmas cannot be judged from
this test window. If a test window that includes them showed a clear shift
on those weeks, that would call for more holiday features, such as weeks
to Thanksgiving or post-Christmas indicators.

## Ideas to Improve the Model

Every retail forecasting project has more juice to squeeze. Things to try
next:

1. **Richer calendar features.** Days to the next holiday, _post_-holiday
   indicators, Easter (not one of the four flagged holidays), paydays,
   school calendars, fiscal week of month.
2. **More history features that respect the horizon.** Lags 13 and 26,
   lags 51 and 53 around last year's week, rolling standard deviation (a
   proxy for volatility). Any new lag must still be at least 12 weeks old.
3. **One model per horizon, or a recursive forecast.** Our single model
   uses 12-week-old history even for the first week of the horizon. You
   could train a separate model for each number of weeks ahead (each using
   the freshest history allowed), or forecast one week at a time and feed
   each forecast back in as the next week's lag.
4. **Hierarchical features.** Department-level chain totals or
   store-level chain totals as additional inputs to capture macro
   shifts.
5. **Better treatment of MarkDowns.** Interactions with department, or a
   model trained only on weeks where markdowns were recorded.
6. **Hyperparameter search with time-based validation.** Tune
   `max_depth`, `learning_rate`, and `n_estimators` on rolling-origin
   splits inside the training weeks, never on the test window.
7. **A log target.** Errors on a \$200k department and a \$2k department
   are on very different scales. Training on `log1p(Weekly_Sales)` can
   help the linear model in particular.
8. **Probabilistic forecasts.** Use quantile regression (`xgboost`'s
   `objective="reg:quantileerror"`) to produce prediction intervals,
   which inventory teams care about even more than point forecasts.
9. **Specialized libraries.** Try `sktime`, `statsforecast`, or `darts`
   for purpose-built forecasting workflows once you understand the basics.

:::{tip} Closing the loop with the business
A forecast is only valuable if it changes a decision. Before tuning
further, ask the inventory or merchandising team **what error tolerance
they need by store and department**: sometimes a 5% MAE improvement is
worth a quarter of effort, and sometimes it would not change the order
quantities at all.
:::

## Summary

- We framed the task as a **12-week-ahead forecast** from one forecast
  origin (3 August 2012). Every history feature looks back at least 12
  weeks, lags are computed on a complete weekly calendar, and missing lags
  are filled from training weeks only.
- Of the two naive baselines, **seasonal naive** (same week last year)
  beat **persistence** (last observed week) overall. Persistence was
  competitive only in the first two weeks of the horizon.
- **XGBoost** beat both baselines on MAE, RMSE, and WMAE, with an MAE
  about 16% below the seasonal naive forecast. **Ridge regression** did
  not beat the seasonal naive forecast on MAE or WMAE.
- XGBoost relied mostly on last year's same week and the recent sales
  level of each series.
- The test window contains one holiday week, Labor Day. These results say
  little about Thanksgiving or Christmas, the weeks that matter most for
  inventory and staffing.

**What this means for planners.** An MAE of about \$1,400 is an average
over departments of very different sizes. Before deciding whether the
forecast is good enough for ordering or staffing, compare the error with
each department's own weekly sales. And before relying on the model for
the holiday season, re-test it on a window that includes November and
December.